# Notebook 06 - Results Writer v1 POC


In [ ]:

# ============================================================
# Notebook 06 - Results Writer
# Agent Evaluation Framework v1 POC
# ============================================================

try:
    run_id
except NameError:
    run_id = "RUN-MANUAL-TEST"

try:
    environment
except NameError:
    environment = "dev"

try:
    poc_mode
except NameError:
    poc_mode = "false"
poc_mode = str(poc_mode).lower()

import importlib.util
import json
import re
import types
from pathlib import Path

from pyspark.sql import functions as F

assert spark is not None, "Spark session not available."
from notebookutils import mssparkutils

LAKEHOUSE_NAME = "jacks_lakehouse"
ONELAKE_WORKSPACE_ID = "d9e51304-2b8a-4e62-b689-922e79fd76b4"
ONELAKE_LAKEHOUSE_ID = "537823c4-b83a-4a9b-9444-6039d55a4b9e"
LAKEHOUSE_DEFAULT_FILES_ROOT = "/lakehouse/default/Files"
LAKEHOUSE_FILES_ABFSS_ROOT = f"abfss://{ONELAKE_WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{ONELAKE_LAKEHOUSE_ID}/Files"
LAKEHOUSE_FILES_ROOT = LAKEHOUSE_FILES_ABFSS_ROOT
BASE_FILES_PATH = f"{LAKEHOUSE_FILES_ROOT}/agent_eval"
BASE_FILES_ABFSS_PATH = f"{LAKEHOUSE_FILES_ABFSS_ROOT}/agent_eval"
CONFIG_PATH = f"{BASE_FILES_PATH}/config"
SHARED_PATH = f"{BASE_FILES_PATH}/shared"
ALERT_CONFIG_YAML_PATH = f"{CONFIG_PATH}/alert_config.yaml"
SECRETS_YAML_PATH = f"{CONFIG_PATH}/secrets.yaml"
AUTH_UTILS_PATH = f"{SHARED_PATH}/auth_utils.py"

CURRENT_RUN_CASES_TABLE = "agent_eval_current_run_cases"
RESPONSES_TABLE = "agent_eval_agent_responses_staging"
DETERMINISTIC_RESULTS_TABLE = "agent_eval_deterministic_results"
RAGAS_SCORES_TABLE = "agent_eval_ragas_scores"
MICROSOFT_EVAL_SCORES_TABLE = "agent_eval_microsoft_eval_scores"
RESULTS_TABLE = "agent_eval_results"
EVIDENCE_TABLE = "agent_eval_evidence"
ALERTS_TABLE = "agent_eval_alerts"
LATEST_RESULTS_VIEW = "agent_eval_last_run_summary"
RUNS_TABLE = "agent_eval_runs"


def load_shared_module(path, module_name):
    if str(path).startswith("abfss://"):
        if not mssparkutils.fs.exists(path):
            raise FileNotFoundError(f"Shared utility module not found: {path}")
        module = types.ModuleType(module_name)
        module.__file__ = path
        exec(mssparkutils.fs.head(path, 20 * 1024 * 1024), module.__dict__)
        return module
    if not Path(path).is_file():
        raise FileNotFoundError(f"Shared utility module not found: {path}")
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


auth_utils = load_shared_module(AUTH_UTILS_PATH, "agent_eval_auth_utils")
alert_config = auth_utils.load_yaml(ALERT_CONFIG_YAML_PATH, {"alerts": {}})


def is_enabled(value):
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "enabled"}


def send_webhook_alerts(alert_rows):
    alerts_cfg = (alert_config or {}).get("alerts", {})
    webhook_cfg = alerts_cfg.get("webhook", {})
    if not is_enabled(webhook_cfg.get("enabled", False)):
        return
    url = auth_utils.get_config_value(webhook_cfg, "url_key", "file", SECRETS_YAML_PATH)
    if not url or "PASTE_" in url:
        print("Webhook alerting enabled but webhook URL is not configured")
        return
    max_rows = int(webhook_cfg.get("max_rows", 20))
    safe_rows = json.loads(json.dumps(alert_rows[:max_rows], default=str))
    payload = {
        "title": "Agent evaluation alert",
        "run_id": run_id,
        "environment": environment,
        "alert_count": len(alert_rows),
        "alerts": safe_rows,
    }
    response = auth_utils.request_with_retries("post", url, json=payload, timeout=30)
    response.raise_for_status()
    print(f"Webhook alert sent. rows={len(alert_rows)}")


def run_df(table_name):
    return spark.table(table_name).filter(F.col("run_id") == run_id)


cases = run_df(CURRENT_RUN_CASES_TABLE)
responses = run_df(RESPONSES_TABLE)
det = run_df(DETERMINISTIC_RESULTS_TABLE)
ragas = run_df(RAGAS_SCORES_TABLE)
ms = run_df(MICROSOFT_EVAL_SCORES_TABLE)

base = cases.select(
    "run_id", "test_id", "agent_id", "category", "suite", "frequency", "severity",
    "question", "test_origin", "ms_test_set_id", "ms_test_case_id"
)

response_summary = responses.select(
    "run_id", "test_id", "agent_id", "status", "agent_response", "latency_ms", "error_type"
)
base = base.join(response_summary, ["run_id", "test_id", "agent_id"], "left")

det_summary = det.groupBy("run_id", "test_id", "agent_id").agg(
    F.max(F.when((F.col("passed") == False) & (F.col("severity") == "critical"), F.lit(1)).otherwise(F.lit(0))).alias("critical_rule_failed"),
    F.sum(F.when(F.col("passed") == False, F.lit(1)).otherwise(F.lit(0))).alias("deterministic_fail_count"),
)
base = base.join(det_summary, ["run_id", "test_id", "agent_id"], "left")

base = base.join(
    ragas.select("run_id", "test_id", "agent_id", "ragas_passed", "faithfulness_score", "answer_relevancy_score", "source_version_hash"),
    ["run_id", "test_id", "agent_id"],
    "left",
)

ms_summary = ms.groupBy("run_id", "agent_id", "ms_test_case_id").agg(
    F.min(F.when(F.col("metric_passed") == False, F.lit(0)).otherwise(F.lit(1))).alias("ms_eval_pass_int"),
    F.first("ms_test_set_id", ignorenulls=True).alias("ms_score_test_set_id"),
    F.count("*").alias("ms_metric_count"),
)
ms_join = (
    ms_summary
    .withColumnRenamed("run_id", "ms_run_id")
    .withColumnRenamed("agent_id", "ms_agent_id")
    .withColumnRenamed("ms_test_case_id", "join_ms_test_case_id")
)
base = base.join(
    ms_join,
    (base.run_id == F.col("ms_run_id")) &
    (base.agent_id == F.col("ms_agent_id")) &
    ((base.ms_test_case_id == F.col("join_ms_test_case_id")) | (base.test_id == F.col("join_ms_test_case_id"))),
    "left",
).drop("ms_run_id", "ms_agent_id", "join_ms_test_case_id")

base = base.fillna({"critical_rule_failed": 0, "deterministic_fail_count": 0, "ms_metric_count": 0})
base = base.withColumn("ms_eval_passed", F.col("ms_eval_pass_int").cast("boolean"))

evaluator_missing = F.col("ragas_passed").isNull() | F.col("ms_eval_passed").isNull()
strict_evaluator_mode = poc_mode not in {"true", "1", "yes"}

if strict_evaluator_mode:
    missing_verdict = F.lit("WARN")
    missing_reason = F.lit("Evaluator signal missing in strict mode")
else:
    missing_verdict = F.lit("CAPTURED")
    missing_reason = F.lit("Agent response captured; evaluator signals not required in POC mode")

results = base.withColumn(
    "verdict",
    F.when(F.col("status") == "FAILED", F.lit("FAIL"))
     .when(F.col("critical_rule_failed") == 1, F.lit("FAIL"))
     .when(evaluator_missing, missing_verdict)
     .when((F.col("ragas_passed") == False) & (F.col("ms_eval_passed") == False), F.lit("FAIL"))
     .when(F.col("ragas_passed") != F.col("ms_eval_passed"), F.lit("WARN"))
     .otherwise(F.lit("PASS"))
).withColumn(
    "verdict_reason",
    F.when(F.col("status") == "FAILED", F.concat(F.lit("Agent call failed: "), F.coalesce(F.col("error_type"), F.lit("unknown"))))
     .when(F.col("critical_rule_failed") == 1, F.lit("Critical deterministic rule failed"))
     .when(evaluator_missing, missing_reason)
     .when((F.col("ragas_passed") == False) & (F.col("ms_eval_passed") == False), F.lit("RAGAS and Microsoft Evaluation both failed"))
     .when(F.col("ragas_passed") != F.col("ms_eval_passed"), F.lit("RAGAS and Microsoft Evaluation disagree - human review"))
     .otherwise(F.lit("All required evaluator gates passed"))
).withColumn(
    "major_rule_warning",
    (F.col("verdict") == "PASS") & (F.col("deterministic_fail_count") > 0)
).withColumn(
    "verdict",
    F.when(F.col("major_rule_warning"), F.lit("WARN")).otherwise(F.col("verdict"))
).withColumn(
    "verdict_reason",
    F.when(F.col("major_rule_warning"), F.lit("Major deterministic rule failed - human review")).otherwise(F.col("verdict_reason"))
).drop("major_rule_warning").withColumn("written_at", F.current_timestamp())

results.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(RESULTS_TABLE)

results.select(
    "run_id", "test_id", "agent_id", "verdict", "verdict_reason",
    "faithfulness_score", "answer_relevancy_score", "ms_eval_passed",
    "source_version_hash", "deterministic_fail_count", "written_at",
).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(EVIDENCE_TABLE)

alerts = (
    results
    .filter(F.col("verdict").isin("WARN", "FAIL"))
    .select("run_id", "test_id", "agent_id", "verdict", "verdict_reason", "written_at")
    .withColumn("alert_status", F.lit("OPEN"))
    .withColumn("alert_channel", F.lit("delta_table"))
)
alerts.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(ALERTS_TABLE)
alert_rows = [
    r.asDict()
    for r in spark.table(ALERTS_TABLE)
        .filter((F.col("run_id") == run_id) & (F.col("alert_status") == "OPEN"))
        .select("run_id", "test_id", "agent_id", "verdict", "verdict_reason", "alert_channel", "written_at")
        .collect()
]
print(f"Alerts written to {ALERTS_TABLE}: {len(alert_rows)}")
if alert_rows:
    send_webhook_alerts(alert_rows)

spark.sql(f"""
CREATE OR REPLACE VIEW {LATEST_RESULTS_VIEW} AS
SELECT *
FROM (
    SELECT
        r.*,
        row_number() OVER (
            PARTITION BY r.agent_id, r.test_id
            ORDER BY r.written_at DESC, r.run_id DESC
        ) AS latest_rank
    FROM {RESULTS_TABLE} r
) latest
WHERE latest_rank = 1
""")
print(f"Latest results view refreshed: {LATEST_RESULTS_VIEW}")

result_count = spark.table(RESULTS_TABLE).filter(F.col("run_id") == run_id).count()
if result_count <= 0:
    raise RuntimeError("Results writer produced no rows")

summary = spark.table(RESULTS_TABLE).filter(F.col("run_id") == run_id).groupBy("verdict").count().collect()
for row in summary:
    print(f"{row['verdict']}: {row['count']}")

try:
    if not re.fullmatch(r"[A-Za-z0-9_.:-]+", run_id):
        raise ValueError(f"Unsafe run_id for SQL update: {run_id}")
    spark.sql(f"UPDATE {RUNS_TABLE} SET status = 'COMPLETED', run_completed_at = current_timestamp() WHERE run_id = '{run_id}'")
except Exception as exc:
    print(f"Run metadata update skipped: {str(exc)[:300]}")

try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit("PASS")
except ImportError:
    pass
